In [2]:
df_train = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_train.parquet", engine="pyarrow")
df_test = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_test.parquet", engine="pyarrow")
df_oot = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oot.parquet", engine="pyarrow")
df_oos = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oos.parquet", engine="pyarrow")

In [3]:
varss = ['attribute4_change_rate',
 'attribute2',
 'attribute8',
 'attribute1_lag_7_max',
 'attribute1_lag_120_max',
 'attribute1_lag_120_mean',
 'attribute5_lag_15_mean',
 'attribute5_lag_60_mean',
 'attribute6_lag_15_std',
 'attribute6_lag_30_min',
 'attribute6_lag_30_std',
 'attribute6_lag_60_min',
 'attribute6_lag_90_std',
 'attribute2_change_rate',
 'attribute2_cumulative_change',
 'attribute4_cumulative_change']

In [4]:
x_train = df_train[varss].copy()
y_train = df_train['failure'].copy()

x_test = df_test[varss]
y_test = df_test['failure']

x_oot = df_oot[varss].copy()
y_oot = df_oot['failure'].copy()

x_oos = df_oos[varss].copy()
y_oos = df_oos['failure'].copy()
    

In [5]:
import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import fbeta_score, precision_recall_curve, average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer

best_params =  {'n_estimators': 235, 'max_depth': 8, 'learning_rate': 0.03396150759743678, 'num_leaves': 44, 'min_child_samples': 36, 'min_child_weight': 0.6092701695282587, 'subsample': 0.8974607246024693, 'colsample_bytree': 0.7806131379276473, 'reg_alpha': 19.354390715869165, 'reg_lambda': 57.03794164972348}

#------------------------------------------------------------
# 3️⃣ Entrenar modelo final con los mejores hiperparámetros
#------------------------------------------------------------
best_model = LGBMClassifier(**best_params, random_state=42)
best_model.fit(x_train, y_train)

[LightGBM] [Info] Number of positive: 83, number of negative: 81133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014737 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3151
[LightGBM] [Info] Number of data points in the train set: 81216, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001022 -> initscore=-6.885004
[LightGBM] [Info] Start training from score -6.885004
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,boosting_type,'gbdt'
,num_leaves,44
,max_depth,8
,learning_rate,0.03396150759743678
,n_estimators,235
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.6092701695282587
,min_child_samples,36


In [7]:
from sklearn.metrics import precision_recall_curve

#------------------------------------------
# Calcular umbral óptimo (best_thr)
#------------------------------------------
y_proba_val = best_model.predict_proba(x_oot)[:, 1]

# Costos operativos
C_FN = 10000.0  # costo por no detectar una falla
C_FP = 100.0    # costo por falsa alarma

prec, rec, thr = precision_recall_curve(y_oot, y_proba_val)

# Calcular FN y FP en función de los umbrales
P = (y_oot == 1).sum()
TP = rec * P
FN = P - TP
FP = (TP / (prec + 1e-12)) - TP

# Calcular costo esperado
cost_expected = C_FN * FN + C_FP * FP

# Índice del costo mínimo
idx_min = np.nanargmin(cost_expected)
best_thr = thr[idx_min] if idx_min < len(thr) else 0.5

print(f"💰 Umbral óptimo por costo esperado: {best_thr:.4f}")
print(f"Precision: {prec[idx_min]:.4f} | Recall: {rec[idx_min]:.4f}")


💰 Umbral óptimo por costo esperado: 0.0086
Precision: 0.3333 | Recall: 0.5000


In [8]:
from sklearn.metrics import (
    precision_score, recall_score, fbeta_score,
    average_precision_score, confusion_matrix
)
import pandas as pd
import numpy as np

def evaluar_modelo_multimuestra(modelo, datasets, best_thr, C_FN=10000.0, C_FP=100.0, beta=2.0):
    """
    Evalúa un modelo binario en múltiples datasets y devuelve un resumen en DataFrame.
    
    Parámetros
    ----------
    modelo : estimator
        Modelo con método predict_proba (LightGBM, XGB, CatBoost, etc.)
    datasets : dict
        Diccionario con pares nombre: (X, y). Ejemplo:
        {
            'Train': (x_train, y_train),
            'Test': (x_test, y_test),
            'OOT': (x_oot, y_oot),
            'OOS': (x_oos, y_oos)
        }
    best_thr : float
        Umbral de probabilidad óptimo.
    C_FN : float, default=10000.0
        Costo por no detectar una falla.
    C_FP : float, default=100.0
        Costo por falsa alarma.
    beta : float, default=2.0
        Peso del recall en F-beta score.

    Retorna
    -------
    pd.DataFrame : reporte de métricas para cada muestra
    """
    reportes = []

    for nombre, (X, y) in datasets.items():
        # Predecir probabilidades y etiquetas
        y_proba = modelo.predict_proba(X)[:, 1]
        y_pred = (y_proba >= best_thr).astype(int)

        # Métricas
        precision = precision_score(y, y_pred, zero_division=0)
        recall = recall_score(y, y_pred, zero_division=0)
        f2 = fbeta_score(y, y_pred, beta=beta, zero_division=0)
        ap = average_precision_score(y, y_proba)

        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        # Costo esperado
        costo = C_FN * fn + C_FP * fp

        reportes.append({
            "Muestra": nombre,
            "Precision": precision,
            "Recall": recall,
            f"F{beta}-score": f2,
            "AUC-PR": ap,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Costo esperado": costo
        })

    df_reporte = pd.DataFrame(reportes).sort_values("Muestra")
    return df_reporte


In [11]:
datasets = {
    "Train": (x_train, y_train),
    "Test": (x_test, y_test),
    "OOT": (x_oot, y_oot),
    "OOS": (x_oos, y_oos)
}

reporte = evaluar_modelo_multimuestra(
    modelo=best_model,
    datasets=datasets,
    best_thr=best_thr,
    C_FN=10000.0,   # costo alto de no detectar falla
    C_FP=100.0,     # costo bajo de falsa alarma
    beta=2.0
)

display(reporte.style.format({
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "AUC-PR": "{:.4f}",
    "Costo esperado": "{:,.0f}"
}))


,Muestra,Precision,Recall,F2.0-score,AUC-PR,TP,FP,FN,TN,Costo esperado
3,OOS,0.0000,0.0000,0.000000,0.0172,0,0,5,11777,"50,000"
2,OOT,0.3333,0.5000,0.454545,0.1670,1,2,1,4001,"10,200"
1,Test,0.2500,0.1333,0.147059,0.1120,2,6,13,10588,"130,600"
0,Train,0.0793,0.1566,0.131048,0.0467,13,151,70,80982,"715,100"
